# Stellar Spectral Synthesis with Korg.jl

This notebook demonstrates how to synthesize stellar spectra using Korg.jl in Julia.

**Main parameters:**
- `Teff`: Effective temperature in Kelvin
- `logg`: Surface gravity (log₁₀ in cgs units)
- `m_H`: Metallicity [M/H] relative to solar
- `wavelengths`: Tuple of (start, end) in Angstroms
- `vmic`: Microturbulence velocity in km/s

In [ ]:
# Load Korg.jl
using Pkg
Pkg.activate("/Users/jdli/Project/Korg.jl")

using Korg
using Plots
gr()  # Use GR backend for plotting

println("✅ Korg.jl loaded successfully")

## Example 1: Solar Spectrum

Synthesize a solar spectrum with standard parameters: Teff=5780K, logg=4.44, [M/H]=0.0

In [ ]:
println("Synthesizing solar spectrum...")

# Simple synthesis using synth()
wavelengths_solar, flux_solar, continuum_solar = synth(
    Teff = 5780.0,           # Effective temperature (K) - Sun
    logg = 4.44,             # Surface gravity (log10 cgs) - Sun
    m_H = 0.0,               # Metallicity [M/H] - Solar
    wavelengths = (5000, 5200)  # Wavelength range (Angstroms)
)

println("\n✅ Synthesis complete!")
println("  Wavelength range: $(wavelengths_solar[1]) - $(wavelengths_solar[end]) Å")
println("  Number of points: $(length(wavelengths_solar))")
println("  Mean flux: $(mean(flux_solar)) erg/s/cm²")
println("  Mean continuum: $(mean(continuum_solar)) erg/s/cm²")

In [ ]:
# Plot solar spectrum
plot(wavelengths_solar, flux_solar, 
     label="Flux", linewidth=0.5, alpha=0.8, color=:blue)
plot!(wavelengths_solar, continuum_solar, 
      label="Continuum", linewidth=1.5, linestyle=:dash, color=:red)
xlabel!("Wavelength (Å)")
ylabel!("Flux (erg/s/cm²)")
title!("Solar Spectrum (Teff=5780K, logg=4.44, [M/H]=0.0)")
plot!(size=(900, 400), legend=:topright, grid=true, gridalpha=0.3)

## Example 2: Cool Giant Star

Synthesize the H-alpha region for a cool, metal-poor giant star.

In [ ]:
println("Synthesizing cool giant spectrum...")

wl_giant, flux_giant, cont_giant = synth(
    Teff = 4500.0,           # Cooler temperature
    logg = 2.0,              # Lower gravity (giant star)
    m_H = -0.5,              # Metal-poor
    wavelengths = (6550, 6575)  # H-alpha region
)

println("\n✅ Synthesis complete!")
println("  Wavelength range: $(wl_giant[1]) - $(wl_giant[end]) Å")
println("  Number of points: $(length(wl_giant))")

In [ ]:
# Plot normalized spectrum
normalized_giant = flux_giant ./ cont_giant

plot(wl_giant, normalized_giant, 
     linewidth=0.8, color=:black, label=false)
vline!([6562.8], linestyle=:dash, color=:red, alpha=0.5, label="H-alpha center")
xlabel!("Wavelength (Å)")
ylabel!("Normalized Flux")
title!("Cool Giant (Teff=4500K, logg=2.0, [M/H]=-0.5) - H-alpha Region")
plot!(size=(900, 400), grid=true, gridalpha=0.3)

## Example 3: Using synthesize() for More Control

The `synthesize()` function provides more detailed control, including custom atmospheres and linelists.

In [ ]:
println("Synthesizing with custom parameters...")

# Load a linelist
linelist_path = "/Users/jdli/Project/Korg.jl/data/linelists/vald_extract_stellar_solar_threshold001.vald"
linelist = read_linelist(linelist_path)
println("Loaded $(length(linelist)) spectral lines")

# Interpolate MARCS atmosphere
atm = interpolate_marcs(6000.0, 4.0, 0.2)  # Metal-rich star

# Set abundances (solar scaled with metallicity)
A_X = format_A_X(0.2)  # [M/H] = +0.2

# Synthesize
result = synthesize(atm, linelist, A_X, (4850, 4870))

println("\n✅ Synthesis complete!")
println("  Wavelength range: $(result.wavelengths[1]) - $(result.wavelengths[end]) Å")
println("  Number of wavelength points: $(length(result.wavelengths))")
println("  Mean flux: $(mean(result.flux)) erg/s/cm²")
println("  Mean continuum: $(mean(result.cntm)) erg/s/cm²")

In [ ]:
# Plot H-beta region
normalized_flux = result.flux ./ result.cntm

plot(result.wavelengths, normalized_flux, 
     linewidth=0.8, color=:blue, label=false)
vline!([4861.3], linestyle=:dash, color=:red, alpha=0.5, label="H-beta center")
xlabel!("Wavelength (Å)")
ylabel!("Normalized Flux")
title!("Metal-Rich Star (Teff=6000K, logg=4.0, [M/H]=+0.2) - H-beta Region")
plot!(size=(900, 400), grid=true, gridalpha=0.3)

## Example 4: Comparing Different Stellar Types

Compare spectra of hot (A-type), solar (G-type), and cool (K-type) stars.

In [ ]:
println("Comparing different stellar types...")

# G-band region (sensitive to temperature)
wl_range = (4300, 4400)

# Hot star (A-type)
println("  Synthesizing A-type star (Teff=8500K)...")
wl_hot, flux_hot, cont_hot = synth(Teff=8500.0, logg=4.0, m_H=0.0, wavelengths=wl_range)

# Solar-type (G-type)
println("  Synthesizing G-type star (Teff=5780K)...")
wl_solar, flux_solar, cont_solar = synth(Teff=5780.0, logg=4.44, m_H=0.0, wavelengths=wl_range)

# Cool star (K-type)
println("  Synthesizing K-type star (Teff=4500K)...")
wl_cool, flux_cool, cont_cool = synth(Teff=4500.0, logg=4.5, m_H=0.0, wavelengths=wl_range)

println("\n✅ All syntheses complete!")

In [ ]:
# Plot comparison
p1 = plot(wl_hot, flux_hot ./ cont_hot, 
          linewidth=0.8, color=:blue, label=false,
          ylabel="Normalized Flux", title="A-type Star (Teff=8500K) - Hot",
          ylims=(0.3, 1.05), grid=true, gridalpha=0.3)

p2 = plot(wl_solar, flux_solar ./ cont_solar, 
          linewidth=0.8, color=:green, label=false,
          ylabel="Normalized Flux", title="G-type Star (Teff=5780K) - Solar",
          ylims=(0.3, 1.05), grid=true, gridalpha=0.3)

p3 = plot(wl_cool, flux_cool ./ cont_cool, 
          linewidth=0.8, color=:red, label=false,
          xlabel="Wavelength (Å)", ylabel="Normalized Flux", 
          title="K-type Star (Teff=4500K) - Cool",
          ylims=(0.3, 1.05), grid=true, gridalpha=0.3)

plot(p1, p2, p3, layout=(3, 1), size=(900, 700),
     plot_title="Comparison of Stellar Spectra (G-band Region)")

## Example 5: Metallicity Effects

Explore how metallicity affects spectral line strength.

In [ ]:
println("Comparing metallicity effects...")

# Fixed stellar parameters, varying metallicity
Teff_fixed = 5500.0
logg_fixed = 4.0
wl_range = (5160, 5180)  # Mg I triplet region

metallicities = [-1.0, -0.5, 0.0, 0.3]
spectra = Dict()

for mh in metallicities
    println("  Synthesizing [M/H]=$(mh)...")
    wl, flux, cont = synth(Teff=Teff_fixed, logg=logg_fixed, m_H=mh, wavelengths=wl_range)
    spectra[mh] = (wl=wl, flux=flux, cont=cont)
end

println("\n✅ All syntheses complete!")

In [ ]:
# Plot metallicity comparison
colors = [:purple, :blue, :green, :red]

p = plot()
for (i, mh) in enumerate(metallicities)
    data = spectra[mh]
    norm_flux = data.flux ./ data.cont
    plot!(p, data.wl, norm_flux, 
          color=colors[i], linewidth=0.8, alpha=0.8,
          label="[M/H]=$(mh)")
end

xlabel!("Wavelength (Å)")
ylabel!("Normalized Flux")
title!("Metallicity Effects on Spectrum (Teff=$(Teff_fixed)K, logg=$(logg_fixed))")
plot!(size=(900, 450), ylims=(0.4, 1.05), 
      legend=:bottomright, grid=true, gridalpha=0.3)

println("\n📝 Note: Higher metallicity → deeper and more numerous absorption lines")

## Example 6: Custom Microturbulence

Vary microturbulence parameter to see its effect on line saturation.

In [ ]:
println("Comparing microturbulence effects...")

# Load linelist for full control
linelist = read_linelist("/Users/jdli/Project/Korg.jl/data/linelists/vald_extract_stellar_solar_threshold001.vald")
atm = interpolate_marcs(5500.0, 4.0, 0.0)
A_X = format_A_X(0.0)

vmics = [0.5, 1.0, 2.0, 3.0]  # km/s
vmic_spectra = Dict()

for vmic in vmics
    println("  Synthesizing vmic=$(vmic) km/s...")
    result = synthesize(atm, linelist, A_X, (5160, 5180), vmic=vmic)
    vmic_spectra[vmic] = result
end

println("\n✅ All syntheses complete!")

In [ ]:
# Plot microturbulence comparison
colors = [:blue, :green, :orange, :red]

p = plot()
for (i, vmic) in enumerate(vmics)
    result = vmic_spectra[vmic]
    norm_flux = result.flux ./ result.cntm
    plot!(p, result.wavelengths, norm_flux, 
          color=colors[i], linewidth=0.8, alpha=0.8,
          label="vmic=$(vmic) km/s")
end

xlabel!("Wavelength (Å)")
ylabel!("Normalized Flux")
title!("Microturbulence Effects on Line Profiles")
plot!(size=(900, 450), ylims=(0.4, 1.05), 
      legend=:bottomright, grid=true, gridalpha=0.3)

println("\n📝 Note: Higher microturbulence → broader, shallower lines (desaturation effect)")

## Example 7: Save Synthetic Spectrum

Save a synthetic spectrum to a text file for further analysis.

In [ ]:
# Synthesize a spectrum to save
println("Generating spectrum to save...")
wl, flux, cont = synth(Teff=5780.0, logg=4.44, m_H=0.0, wavelengths=(5000, 5100))

# Save to file
output_file = "/Users/jdli/Project/Korg.jl/jorg/examples/korg_synthetic_spectrum.txt"

open(output_file, "w") do io
    # Write header
    println(io, "# Synthetic Stellar Spectrum")
    println(io, "# Generated with Korg.jl")
    println(io, "# Stellar Parameters:")
    println(io, "#   Teff = 5780 K")
    println(io, "#   logg = 4.44")
    println(io, "#   [M/H] = 0.0")
    println(io, "# Columns: wavelength(Å) flux(erg/s/cm²) continuum(erg/s/cm²) normalized_flux")
    
    # Write data
    for i in 1:length(wl)
        normalized = flux[i] / cont[i]
        println(io, "$(wl[i])\t$(flux[i])\t$(cont[i])\t$(normalized)")
    end
end

println("\n✅ Spectrum saved to: $(output_file)")
println("  Number of points: $(length(wl))")

## Summary

This notebook demonstrated:

1. **Basic synthesis** with `synth()` for quick spectral generation
2. **Cool giant stars** with different atmospheric parameters
3. **Using `synthesize()`** with custom atmospheres and linelists
4. **Comparing stellar types** (A, G, K spectral classes)
5. **Metallicity effects** on line strengths
6. **Microturbulence effects** on line profiles
7. **Saving spectra** to files for further analysis

**Key Korg.jl Functions:**
- `synth()`: Simple interface for quick synthesis
- `synthesize()`: Full control with custom parameters
- `read_linelist()`: Load VALD/Kurucz linelists
- `interpolate_marcs()`: Get MARCS model atmosphere
- `format_A_X()`: Set up abundance patterns

**Physics Features:**
- 277-species chemical equilibrium
- MARCS model atmospheres (56 layers)
- Complete continuum opacity (H⁻, Thomson, Rayleigh, metal bf/ff)
- Line absorption with exact Voigt profiles
- Radiative transfer with anchored optical depth method